# Cell 1: Markdown title
# Preprocessing Pipeline for YOLO Training

## Preprocessing Steps Explained

- **Letterbox Resizing**  
  Resize each image to a fixed square size (e.g. 640×640) without distortion by scaling to fit and padding the shorter edges. Ensures all inputs share the same dimensions while preserving object shapes and centering content.

- **CLAHE (Contrast-Limited Adaptive Histogram Equalization)**  
  Enhances local contrast by applying histogram equalization on small image tiles, then clipping extreme amplification to avoid noise blow-up. Makes faint features (like flagellar motors) more visible in low-contrast tomogram slices.

- **NLMeans Denoising**  
  Reduces speckle and random noise by averaging each patch with similar patches found across the image, preserving textures and edges. Particularly effective for the grainy appearance of cryo-ET data.

- **Grayscale Normalization**  
  Stretches pixel intensities to the full 0–255 range, standardizing brightness and contrast across slices. A simple yet powerful step before further enhancement and denoising.

Each of these operations boosts the signal-to-noise and ensures uniform, centered inputs for a YOLO detector, improving both training stability and detection accuracy.```


## References

[Data Preprocessing](https://docs.ultralytics.com/guides/preprocessing_annotated_data/)

[Data Augmentation](https://www.genome.gov/)



In [2]:
import os
import sys
from glob import glob
from tqdm import tqdm
import random
import shutil
import cv2
from collections import defaultdict
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils import resample
import albumentations as A


# Add project root to system path (for relative imports to work)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src import config


In [17]:
# Cell 2: Imports & Configuration

# User parameters
INPUT_DIR    = config.DATASET_DIR           # root of your subfolders (e.g. tomo_xxx)
OUTPUT_DIR   = config.PREPROCESSED_DATASET_DIR
TARGET_SIZE  = (640, 640)                          # (height, width)
EXTS         = [".jpg", ".png", ".tif", ".tiff"]   # supported extensions

# CLAHE & denoise settings
CLAHE_CFG    = {"clip_limit": 2.0, "grid_size": (8, 8)}
DENOISE_CFG  = {"h": 10, "template_size": 7, "search_size": 21}

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [18]:
# Cell 3: Helper Functions (updated to handle raw gray data)

import cv2

def letterbox_resize(img, new_shape=TARGET_SIZE, color=(114,114,114)):
    """Resize+pad to new_shape, keeping aspect ratio (letterbox)."""
    h0, w0 = img.shape[:2]
    r = min(new_shape[0]/h0, new_shape[1]/w0)
    new_unpad = (int(w0*r), int(h0*r))
    dh, dw = new_shape[0] - new_unpad[1], new_shape[1] - new_unpad[0]
    dh, dw = dh/2, dw/2

    img = cv2.resize(img, new_unpad, interpolation=cv2.INTER_LINEAR)
    top, bottom = int(round(dh-0.1)), int(round(dh+0.1))
    left, right = int(round(dw-0.1)), int(round(dw+0.1))
    # if single‐channel, pad with one value; if 3‐channel, pad with color tuple
    if len(img.shape)==2:
        return cv2.copyMakeBorder(img, top, bottom, left, right,
                                  cv2.BORDER_CONSTANT, value=color[0])
    else:
        return cv2.copyMakeBorder(img, top, bottom, left, right,
                                  cv2.BORDER_CONSTANT, value=color)

def apply_clahe(gray, clip_limit, grid_size):
    """Apply CLAHE on a grayscale image."""
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=grid_size)
    return clahe.apply(gray)

def denoise_image(gray, h, template_size, search_size):
    """Denoise grayscale with NLMeans."""
    return cv2.fastNlMeansDenoising(gray, None, h, template_size, search_size)

def preprocess_image(img, 
                     clahe_cfg=CLAHE_CFG, 
                     denoise_cfg=DENOISE_CFG, 
                     target_size=TARGET_SIZE):
    """
    Full pipeline: detect gray→ normalize→ CLAHE→ denoise→ ensure 3ch→ letterbox.
    """
    # 1) If image is BGR (3ch), convert to gray; else assume single‐channel already
    if img.ndim == 3 and img.shape[2] == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img.copy()

    # 2) Normalize to [0,255]
    norm = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX).astype('uint8')

    # 3) CLAHE
    if clahe_cfg:
        norm = apply_clahe(norm, **clahe_cfg)

    # 4) Denoise
    if denoise_cfg:
        norm = denoise_image(norm, **denoise_cfg)

    # 5) Convert back to 3‐channel BGR for downstream consistency
    bgr = cv2.cvtColor(norm, cv2.COLOR_GRAY2BGR)

    # 6) Letterbox resize to target_size
    return letterbox_resize(bgr, new_shape=target_size)


In [20]:
# Cell 4: Batch Processing Function

def process_directory(input_dir, output_dir, exts=EXTS):
    """
    Walks through input_dir, preprocesses each image, and writes
    to output_dir mirroring folder structure.
    """
    total = 0
    for root, _, files in os.walk(input_dir):
        rel = os.path.relpath(root, input_dir)
        out_subdir = os.path.join(output_dir, rel)
        os.makedirs(out_subdir, exist_ok=True)

        for fname in files:
            if not any(fname.lower().endswith(ext) for ext in exts):
                continue
            src = os.path.join(root, fname)
            img = cv2.imread(src)
            if img is None:
                continue

            pre = preprocess_image(img)
            dst = os.path.join(out_subdir, fname)
            cv2.imwrite(dst, pre)
            total += 1

    return total


In [21]:
# Cell 5: Run Preprocessing

print(f"▶️ Preprocessing {INPUT_DIR} → {OUTPUT_DIR} ...")
count = process_directory(INPUT_DIR, OUTPUT_DIR)
print(f"✅ Done! Processed {count} images.")


▶️ Preprocessing /Users/kereminci/Desktop/cms-team/BYU_Locating_Bacterial_Flagellar_Motors_2025/sampled_train → /Users/kereminci/Desktop/cms-team/BYU_Locating_Bacterial_Flagellar_Motors_2025/preprocessed_data ...
✅ Done! Processed 1400 images.


Convert raw sample data to yolo format

In [29]:
# Set random seed for reproducibility
RANDOM_STATE = 42
random.seed(RANDOM_STATE)

# Input paths
data_root    = config.PREPROCESSED_DATASET_DIR
labels_csv   = config.TRAIN_LABELS_PATH

# Output base paths (all under PROJECT_ROOT/data/yolo)
output_base         = os.path.join(config.PROJECT_ROOT, "data", "yolo")
output_images_train = os.path.join(output_base, "images", "train")
output_images_val   = os.path.join(output_base, "images", "val")
output_labels_train = os.path.join(output_base, "labels", "train")
output_labels_val   = os.path.join(output_base, "labels", "val")

for path in (output_images_train, output_images_val, output_labels_train, output_labels_val):
    os.makedirs(path, exist_ok=True)

# Load labels CSV
labels_df = pd.read_csv(labels_csv)

# Build positive_samples: image_path -> list of boxes (x,y,img_w,img_h)
positive_samples = defaultdict(list)
for _, row in labels_df.iterrows():
    if row["Motor axis 0"] < 0:
        continue
    tomo   = row["tomo_id"]
    z      = int(row["Motor axis 0"])
    y, x   = row["Motor axis 1"], row["Motor axis 2"]
    H, W   = row["Array shape (axis 1)"], row["Array shape (axis 2)"]
    img    = os.path.join(data_root, tomo, f"slice_{z:04d}.jpg")
    positive_samples[img].append((x, y, W, H))

# Gather all slice image paths and their tomo_id
records = []
for root, _, files in os.walk(data_root):
    tomo = os.path.basename(root)
    for fname in files:
        if not fname.endswith('.jpg'):
            continue
        path = os.path.join(root, fname)
        # count motors on this slice
        count = len(positive_samples.get(path, []))
        records.append({"img": path, "tomo_id": tomo, "motor_count": count})

images_df = pd.DataFrame(records)

# Group-aware split: stratify by tomogram having any motor
tomos = images_df['tomo_id'].unique()
# Compute group label: 1 if tomo has any motor in any slice, else 0
tomo_labels = images_df.groupby('tomo_id')['motor_count']\
    .sum().gt(0).astype(int)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, val_idx = next(
    splitter.split(images_df, groups=images_df['tomo_id'],
                   y=images_df['tomo_id'].map(tomo_labels))
)
train_df = images_df.iloc[train_idx].reset_index(drop=True)
val_df   = images_df.iloc[val_idx].reset_index(drop=True)

# Balance training set: keep all positives, undersample negatives to 2:1 ratio
pos = train_df[train_df.motor_count > 0]
neg = train_df[train_df.motor_count == 0]
n_keep = min(len(neg), len(pos) * 2)
neg_down = resample(neg, replace=False, n_samples=n_keep, random_state=RANDOM_STATE)
train_balanced = pd.concat([pos, neg_down]).sample(frac=1, random_state=RANDOM_STATE)

# Function to copy images and labels
bbox_size = (32, 32)
def process_rows(df, out_img_dir, out_lbl_dir):
    for row in df.itertuples():
        src = row.img
        tomo = row.tomo_id
        z = int(os.path.splitext(os.path.basename(src))[0].split('_')[-1])
        base = f"{tomo}_slice_{z:04d}"
        dst_img = os.path.join(out_img_dir, f"{base}.jpg")
        dst_lbl = os.path.join(out_lbl_dir, f"{base}.txt")
        os.makedirs(os.path.dirname(dst_img), exist_ok=True)
        os.makedirs(os.path.dirname(dst_lbl), exist_ok=True)
        shutil.copyfile(src, dst_img)
        # write labels
        with open(dst_lbl, 'w') as f:
            for (x, y, W, H) in positive_samples.get(src, []):
                cx = x / W
                cy = y / H
                nw = bbox_size[0] / W
                nh = bbox_size[1] / H
                f.write(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n")

# Generate train/val sets
process_rows(train_balanced, output_images_train, output_labels_train)
process_rows(val_df,         output_images_val,   output_labels_val)

print(f"Train slices: {len(train_balanced)}, Val slices: {len(val_df)}")


Train slices: 6, Val slices: 300


In [30]:

random.seed(42)

# Input paths (now from config)
data_root    = config.PREPROCESSED_DATASET_DIR
labels_csv   = config.TRAIN_LABELS_PATH

# Output base paths (all under PROJECT_ROOT/data/yolo)
output_base         = os.path.join(config.PROJECT_ROOT, "data", "yolo")
output_images_train = os.path.join(output_base, "images", "train")
output_images_val   = os.path.join(output_base, "images", "val")
output_labels_train = os.path.join(output_base, "labels", "train")
output_labels_val   = os.path.join(output_base, "labels", "val")

# Create output directories
for path in (output_images_train, output_images_val, output_labels_train, output_labels_val):
    os.makedirs(path, exist_ok=True)

# Load labels CSV
labels_df = pd.read_csv(labels_csv)

# Build a dict of positive samples: image_path -> (x, y, width, height, img_width, img_height)
positive_samples = {}
for _, row in labels_df.iterrows():
    if row["Motor axis 0"] == -1:  # -1 means no motor (negative sample)
        continue
    tomo_id    = row["tomo_id"]
    z          = int(row["Motor axis 0"])
    y          = row["Motor axis 1"]
    x          = row["Motor axis 2"]
    img_width  = row["Array shape (axis 2)"]
    img_height = row["Array shape (axis 1)"]
    img_name   = f"slice_{z:04d}.jpg"
    img_path   = os.path.join(data_root, tomo_id, img_name)
    positive_samples[img_path] = (x, y, img_width, img_height)

# Gather all image paths
all_images = []
for root, _, files in os.walk(data_root):
    for fname in files:
        if fname.endswith(".jpg"):
            all_images.append(os.path.join(root, fname))

# Shuffle and split
random.shuffle(all_images)
split_idx    = int(len(all_images) * 0.8)
train_images = all_images[:split_idx]
val_images   = all_images[split_idx:]

def process_images(image_list, img_out_dir, lbl_out_dir):
    for img_path in image_list:
        tomo_id = os.path.basename(os.path.dirname(img_path))
        z       = int(os.path.splitext(img_path)[0].split("_")[-1])
        base_fn = f"{tomo_id}_slice_{z:04d}"
        dest_img = os.path.join(img_out_dir,  f"{base_fn}.jpg")
        dest_lbl = os.path.join(lbl_out_dir, f"{base_fn}.txt")

        shutil.copyfile(img_path, dest_img)

        if img_path in positive_samples:
            x, y, w, h = positive_samples[img_path]
            # choose bounding‐box size or derive from data
            bbox_w, bbox_h = 10, 10
            cx = x / w
            cy = y / h
            nw = bbox_w / w
            nh = bbox_h / h
            with open(dest_lbl, "w") as f:
                f.write(f"0 {cx} {cy} {nw} {nh}\n")
        else:
            # negative sample ⇒ empty label
            open(dest_lbl, "w").close()

# Process both sets
process_images(train_images, output_images_train, output_labels_train)
process_images(val_images,   output_images_val,   output_labels_val)

print(f"Dataset created with {len(train_images)} training and {len(val_images)} validation images.") 


Dataset created with 1120 training and 280 validation images.


In [ ]:

# 1) Define your augmentation pipeline
transform = A.Compose([
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, 
                       rotate_limit=15, p=0.8),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, 
                               contrast_limit=0.2, p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.3),
], 
# IMPORTANT: tell it how to handle YOLO bboxes
bbox_params=A.BboxParams(format='yolo',
                         label_fields=['class_labels'],
                         min_visibility=0.2)
)

# 2) Load your YOLO labels (one file per image)
def load_yolo_labels(label_path):
    bboxes, class_labels = [], []
    for line in open(label_path):
        cls, x, y, w, h = map(float, line.split())
        bboxes.append([x, y, w, h])
        class_labels.append(int(cls))
    return bboxes, class_labels

# 3) Augment and save
data_root    = config.YOLO_DATA_DIR
output_root = config.AUGMENTED_YOLO_DATA

IMAGES_DIR = os.path.join(data_root, "images", "train")
LABELS_DIR = os.path.join(data_root, "labels", "train")

os.path.join(output_root, "images", "train")
os.path.join(output_root, "labels", "train")

OUT_IMAGES = os.path.join(output_root, "images", "train")
OUT_LABELS = os.path.join(output_root, "labels", "train")

os.makedirs(OUT_IMAGES, exist_ok=True)
os.makedirs(OUT_LABELS, exist_ok=True)

for img_name in tqdm(os.listdir(IMAGES_DIR)):
    img_path = os.path.join(IMAGES_DIR, img_name)
    label_path = os.path.join(LABELS_DIR, img_name.replace('.jpg','.txt'))

    # read
    img = cv2.imread(img_path)
    bboxes, class_labels = load_yolo_labels(label_path)

    # apply N random augmentations per image
    for i in range(3):  # generate 3 aug versions
        transformed = transform(image=img, bboxes=bboxes, 
                                class_labels=class_labels)
        aug_img  = transformed['image']
        aug_bboxes = transformed['bboxes']
        aug_labels = transformed['class_labels']

        # save image
        out_img = os.path.join(OUT_IMAGES, f"{os.path.splitext(img_name)[0]}_aug{i}.jpg")
        cv2.imwrite(out_img, aug_img)

        # save labels in YOLO format
        out_lbl = os.path.join(OUT_LABELS, f"{os.path.splitext(img_name)[0]}_aug{i}.txt")
        with open(out_lbl, 'w') as f:
            for cls, (x,y,w,h) in zip(aug_labels, aug_bboxes):
                f.write(f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")


/var/folders/3c/s9rrk0m14p5b80qglgftm0900000gn/T/ipykernel_56963/103809204.py:5: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
/var/folders/3c/s9rrk0m14p5b80qglgftm0900000gn/T/ipykernel_56963/103809204.py:10: UserWarning: Argument(s) 'alpha_affine' are not valid for transform ElasticTransform
  A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.3),
100%|██████████| 1120/1120 [01:38<00:00, 11.43it/s]
